# LeetCode 1071 — Greatest Common Divisor of Strings bằng Python

**Link đề:** https://leetcode.com/problems/greatest-common-divisor-of-strings/
**Độ khó:** Easy · **Chủ đề:** Math, String, Euclidean Algorithm, GCD

```python
class Solution:
    def gcdOfStrings(self, str1: str, str2: str) -> str:
        ...
```

## Mục tiêu học
1. Hiểu định nghĩa "chuỗi `t` **chia hết** chuỗi `s`": `s = t + t + ... + t`.
2. Ôn **ước chung lớn nhất** (GCD) của hai số và **thuật toán Euclid**; dùng `math.gcd`.
3. Viết lời giải thử mọi tiền tố (baseline), rồi rút ra 2 lời giải đẹp:
   - Kiểm tra `str1 + str2 == str2 + str1` rồi lấy tiền tố dài `gcd(len1, len2)`.
   - Euclid **trực tiếp trên chuỗi** (đệ quy cắt tiền tố).
4. Trace, chứng minh tính đúng, phân tích Time/Space, tự làm biến thể.

**Cách học:** đọc markdown → đoán output của cell code kế tiếp → chạy → so sánh. Các cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài — "ước chung lớn nhất" của hai chuỗi

Với hai chuỗi `s` và `t`, ta nói **"`t` chia hết `s`"** nếu và chỉ nếu `s = t + t + ... + t` (nối `t` với chính nó **một hoặc nhiều** lần).

Cho `str1`, `str2`, trả về chuỗi `x` **dài nhất** sao cho `x` chia hết **cả** `str1` và `str2`. Không có thì trả `""`.

| `str1` | `str2` | Output | Giải thích |
|---|---|---|---|
| `"ABCABC"` | `"ABC"` | `"ABC"` | `ABCABC = ABC·2`, `ABC = ABC·1` |
| `"ABABAB"` | `"ABAB"` | `"AB"` | `AB·3` và `AB·2` |
| `"LEET"` | `"CODE"` | `""` | không có khối chung |
| `"AAAAAB"` | `"AAA"` | `""` | `AAAAAB` không là lặp của khối nào chia hết `AAA` |

**Constraints:** `1 <= str1.length, str2.length <= 1000`; chỉ gồm chữ in hoa.

In [1]:
t = "AB"
print(t * 3, "==", "ABABAB", "->", t * 3 == "ABABAB")
print(t * 2, "==", "ABAB", "->", t * 2 == "ABAB")

ABABAB == ABABAB -> True
ABAB == ABAB -> True


## 2. Nền tảng cần biết

### 2.1. Nhân chuỗi với số nguyên: `t * k`

`t * k` nối `t` với chính nó `k` lần. `t * 0` là chuỗi rỗng. Đây là cách viết gọn của `t + t + ... + t`.

In [2]:
print("AB" * 1, "AB" * 3, repr("AB" * 0))

AB ABABAB ''


### 2.2. Khi nào `t` chia hết `s`? — hai điều kiện

1. `len(s)` chia hết cho `len(t)`: `len(s) % len(t) == 0`.
2. Lặp `t` đúng `len(s) // len(t)` lần ra `s`.

Hàm kiểm tra:

In [3]:
def divides(t: str, s: str) -> bool:
    if len(s) % len(t) != 0:
        return False
    return t * (len(s) // len(t)) == s

print(divides("AB", "ABABAB"), divides("ABAB", "ABABAB"), divides("ABC", "ABCABC"))

True False True


### 2.3. GCD của hai số và thuật toán Euclid

**GCD** (Greatest Common Divisor — ước chung lớn nhất) của `a`, `b` là số lớn nhất chia hết cả hai. `gcd(6, 4) = 2`.

**Euclid:** `gcd(a, b) = gcd(b, a % b)`, dừng khi `b = 0` thì đáp án là `a`.

Ví dụ `gcd(6, 4)`: `(6, 4) → (4, 2) → (2, 0)` → `2`.

Vì sao đúng? Mọi ước chung của `a`, `b` cũng chia hết `a - b` (và `a % b` = `a` trừ đi nhiều lần `b`), và ngược lại.

In [4]:
def gcd_euclid(a: int, b: int) -> int:
    while b:
        print(f"  gcd({a}, {b}) -> gcd({b}, {a % b})")
        a, b = b, a % b
    return a

print("kết quả:", gcd_euclid(6, 4))
print("kết quả:", gcd_euclid(1000, 750))

  gcd(6, 4) -> gcd(4, 2)
  gcd(4, 2) -> gcd(2, 0)
kết quả: 2
  gcd(1000, 750) -> gcd(750, 250)
  gcd(750, 250) -> gcd(250, 0)
kết quả: 250


### 2.4. `math.gcd` có sẵn

Thư viện chuẩn có `math.gcd(a, b)`. Ta dùng nó trong code nộp cho gọn.

Cú pháp `a, b = b, a % b` là **gán đồng thời** (tuple unpacking): vế phải được tính xong **trước**, rồi mới gán — không cần biến tạm.

In [5]:
import math
print(math.gcd(6, 4), math.gcd(1000, 750), math.gcd(7, 3), math.gcd(5, 5))
a, b = 6, 4
a, b = b, a % b
print(a, b)     # 4 2

2 250 1 5
4 2


## 3. Cách 1 — Thử mọi tiền tố, từ dài đến ngắn (baseline)

Nếu `x` chia hết `str1` thì `str1` bắt đầu bằng `x` ⇒ `x` phải là **tiền tố** (prefix) của cả hai chuỗi. Gợi ý của đề cũng nói vậy.

Thuật toán:
1. Cho `L` chạy từ `min(len1, len2)` **giảm dần** về 1.
2. `cand = str1[:L]`. Nếu `cand` chia hết cả hai → trả `cand` (đây là dài nhất vì thử từ dài xuống).
3. Hết vòng → `""`.

In [6]:
def gcd_strings_brute(str1: str, str2: str) -> str:
    for L in range(min(len(str1), len(str2)), 0, -1):
        cand = str1[:L]
        if divides(cand, str1) and divides(cand, str2):
            return cand
    return ""

print(repr(gcd_strings_brute("ABCABC", "ABC")))
print(repr(gcd_strings_brute("ABABAB", "ABAB")))
print(repr(gcd_strings_brute("LEET", "CODE")))
print(repr(gcd_strings_brute("AAAAAB", "AAA")))

'ABC'
'AB'
''
''


### `range(start, stop, -1)` — đếm lùi

`range(5, 0, -1)` cho `5, 4, 3, 2, 1` (dừng **trước** `0`). Bước âm giúp thử độ dài lớn trước.

In [7]:
print(list(range(5, 0, -1)))
print(list(range(3, 0, -1)))

[5, 4, 3, 2, 1]
[3, 2, 1]


### Trace cách 1 với `"ABABAB"`, `"ABAB"`

In [8]:
def trace_brute(str1: str, str2: str) -> str:
    for L in range(min(len(str1), len(str2)), 0, -1):
        cand = str1[:L]
        d1, d2 = divides(cand, str1), divides(cand, str2)
        print(f"L={L}: cand={cand!r:8} chia hết str1? {d1!s:5} | str2? {d2}")
        if d1 and d2:
            return cand
    return ""

print("->", repr(trace_brute("ABABAB", "ABAB")))

L=4: cand='ABAB'   chia hết str1? False | str2? True
L=3: cand='ABA'    chia hết str1? False | str2? False
L=2: cand='AB'     chia hết str1? True  | str2? True
-> 'AB'


## 4. Nút thắt & insight

Cách 1 tốn `O(min(n1, n2) · (n1 + n2))` — ổn với 1000 ký tự, nhưng ta muốn hiểu **cấu trúc** bài toán.

**Insight 1 — độ dài:** nếu `x` chia hết cả hai thì `len(x)` chia hết `len1` và `len2` ⇒ `len(x)` là **ước chung** của hai độ dài. Muốn dài nhất → thử ứng viên dài `gcd(len1, len2)`.

**Insight 2 — điều kiện tồn tại:** nếu `str1 = x·p` và `str2 = x·q` thì

```text
str1 + str2 = x·(p+q) = str2 + str1
```

Tức là **ghép theo hai thứ tự cho cùng kết quả**. Ngược lại (có thể chứng minh): nếu `str1 + str2 == str2 + str1` thì cả hai đều là lặp của một khối chung.

⇒ **Thuật toán:** nếu `str1 + str2 != str2 + str1` → `""`; ngược lại → `str1[:gcd(len1, len2)]`.

In [9]:
for s1, s2 in [("ABCABC", "ABC"), ("ABABAB", "ABAB"), ("LEET", "CODE"), ("AAAAAB", "AAA")]:
    print(f"{s1!r:9} + {s2!r:7}: s1+s2={s1 + s2!r:14} s2+s1={s2 + s1!r:14} bằng? {s1 + s2 == s2 + s1}")

'ABCABC'  + 'ABC'  : s1+s2='ABCABCABC'    s2+s1='ABCABCABC'    bằng? True
'ABABAB'  + 'ABAB' : s1+s2='ABABABABAB'   s2+s1='ABABABABAB'   bằng? True
'LEET'    + 'CODE' : s1+s2='LEETCODE'     s2+s1='CODELEET'     bằng? False
'AAAAAB'  + 'AAA'  : s1+s2='AAAAABAAA'    s2+s1='AAAAAAAAB'    bằng? False


## 5. Cách 2 (lời giải chính) — Kiểm tra giao hoán + GCD độ dài

**Đoạn 1:** `if str1 + str2 != str2 + str1: return ""`.

**Đoạn 2:** `g = math.gcd(len(str1), len(str2))`.

**Đoạn 3:** `return str1[:g]`.

In [10]:
def gcd_strings_concat(str1: str, str2: str) -> str:
    if str1 + str2 != str2 + str1:        # Đoạn 1: không có khối chung
        return ""
    g = math.gcd(len(str1), len(str2))    # Đoạn 2
    return str1[:g]                       # Đoạn 3

for s1, s2 in [("ABCABC", "ABC"), ("ABABAB", "ABAB"), ("LEET", "CODE"), ("AAAAAB", "AAA")]:
    print(repr(s1), repr(s2), "->", repr(gcd_strings_concat(s1, s2)))

'ABCABC' 'ABC' -> 'ABC'
'ABABAB' 'ABAB' -> 'AB'
'LEET' 'CODE' -> ''
'AAAAAB' 'AAA' -> ''


### Trace cách 2

| `str1` | `str2` | `s1+s2 == s2+s1`? | `gcd(len)` | Kết quả |
|---|---|:---:|:---:|---|
| `ABCABC` (6) | `ABC` (3) | có | 3 | `ABC` |
| `ABABAB` (6) | `ABAB` (4) | có | 2 | `AB` |
| `LEET` (4) | `CODE` (4) | không | — | `""` |
| `AAAAAB` (6) | `AAA` (3) | không | — | `""` |

Ví dụ thú vị: `"AAAAAA"` và `"AAAA"` — giao hoán đúng, `gcd(6, 4) = 2` → `"AA"`. Dù `"A"` cũng chia hết cả hai, `"AA"` dài hơn.

In [11]:
def trace_concat(str1: str, str2: str) -> str:
    a, b2 = str1 + str2, str2 + str1
    print(f"str1+str2 = {a!r}\nstr2+str1 = {b2!r}\nbằng nhau? {a == b2}")
    if a != b2:
        print("-> không có ước chung, trả ''")
        return ""
    g = math.gcd(len(str1), len(str2))
    print(f"gcd({len(str1)}, {len(str2)}) = {g} -> trả {str1[:g]!r}")
    return str1[:g]

trace_concat("AAAAAA", "AAAA"); print()
trace_concat("AAAAAB", "AAA")

str1+str2 = 'AAAAAAAAAA'
str2+str1 = 'AAAAAAAAAA'
bằng nhau? True
gcd(6, 4) = 2 -> trả 'AA'

str1+str2 = 'AAAAABAAA'
str2+str1 = 'AAAAAAAAB'
bằng nhau? False
-> không có ước chung, trả ''


''

## 6. Cách 3 — Euclid trực tiếp trên chuỗi

Bắt chước `gcd(a, b) = gcd(a - b, b)` (Euclid dạng trừ):

- Giả sử `len(str1) >= len(str2)`. Nếu `str1` **không** bắt đầu bằng `str2` → không có ước chung → `""`.
- Ngược lại cắt `str2` khỏi đầu `str1`: `gcd(str1[len(str2):], str2)`.
- Khi một chuỗi rỗng, chuỗi còn lại là đáp án (giống `gcd(a, 0) = a`).

Vì sao cắt được? Nếu `x` chia hết cả `str1 = x·p` và `str2 = x·q` (`p >= q`) thì `str1[len(str2):] = x·(p-q)` — vẫn chia hết bởi `x`.

In [12]:
def gcd_strings_euclid(str1: str, str2: str) -> str:
    if len(str1) < len(str2):
        str1, str2 = str2, str1             # đảm bảo str1 dài hơn
    if not str2:
        return str1                         # gcd(a, 0) = a
    if not str1.startswith(str2):
        return ""
    return gcd_strings_euclid(str1[len(str2):], str2)

for s1, s2 in [("ABCABC", "ABC"), ("ABABAB", "ABAB"), ("LEET", "CODE"), ("AAAAAB", "AAA")]:
    print(repr(s1), repr(s2), "->", repr(gcd_strings_euclid(s1, s2)))

'ABCABC' 'ABC' -> 'ABC'
'ABABAB' 'ABAB' -> 'AB'
'LEET' 'CODE' -> ''
'AAAAAB' 'AAA' -> ''


In [13]:
def trace_euclid(str1: str, str2: str, depth: int = 0) -> str:
    pad = "  " * depth
    if len(str1) < len(str2):
        str1, str2 = str2, str1
    print(f"{pad}gcd({str1!r}, {str2!r})")
    if not str2:
        print(f"{pad}-> chuỗi ngắn rỗng, đáp án {str1!r}")
        return str1
    if not str1.startswith(str2):
        print(f"{pad}-> {str1!r} không bắt đầu bằng {str2!r}: trả ''")
        return ""
    return trace_euclid(str1[len(str2):], str2, depth + 1)

trace_euclid("ABABAB", "ABAB"); print()
trace_euclid("AAAAAB", "AAA")

gcd('ABABAB', 'ABAB')
  gcd('ABAB', 'AB')
    gcd('AB', 'AB')
      gcd('AB', '')
      -> chuỗi ngắn rỗng, đáp án 'AB'

gcd('AAAAAB', 'AAA')
  gcd('AAB', 'AAA')
  -> 'AAB' không bắt đầu bằng 'AAA': trả ''


''

### Đệ quy (recursion) ngắn gọn

Hàm gọi lại chính nó với input **nhỏ hơn**, và có **điều kiện dừng** (`not str2` hoặc không khớp tiền tố). Mỗi lần gọi, tổng độ dài giảm ít nhất `len(str2) >= 1`, nên chắc chắn dừng. Nhưng trường hợp xấu như `"A"*1000` và `"A"` cần khoảng 1000 lần gọi lồng nhau — chạm giới hạn đệ quy mặc định (khoảng 1000) của Python và có thể báo `RecursionError`! Vì vậy bản nộp nên viết **vòng lặp**.

In [14]:
import sys
print("giới hạn đệ quy mặc định:", sys.getrecursionlimit())

def gcd_strings_euclid_iter(str1: str, str2: str) -> str:
    while str2:
        if len(str1) < len(str2):
            str1, str2 = str2, str1
            continue
        if not str1.startswith(str2):
            return ""
        str1 = str1[len(str2):]
    return str1

print(repr(gcd_strings_euclid_iter("A" * 1000, "A")))      # 'A', không lỗi đệ quy
print(repr(gcd_strings_euclid_iter("ABABAB", "ABAB")))

giới hạn đệ quy mặc định: 1000
'A'
'AB'


## 7. Code nộp LeetCode

### Bản giao hoán + GCD (lời giải chính)

In [15]:
import math

class Solution:
    def gcdOfStrings(self, str1: str, str2: str) -> str:
        if str1 + str2 != str2 + str1:
            return ""
        return str1[:math.gcd(len(str1), len(str2))]

sol = Solution()
assert sol.gcdOfStrings("ABCABC", "ABC") == "ABC"
assert sol.gcdOfStrings("ABABAB", "ABAB") == "AB"
assert sol.gcdOfStrings("LEET", "CODE") == ""
assert sol.gcdOfStrings("AAAAAB", "AAA") == ""
print("Solution PASS")

Solution PASS


### Bản thử tiền tố (dễ tự nghĩ trong phòng thi)

In [16]:
class SolutionBrute:
    def gcdOfStrings(self, str1: str, str2: str) -> str:
        n1, n2 = len(str1), len(str2)
        for L in range(min(n1, n2), 0, -1):
            if n1 % L == 0 and n2 % L == 0:          # lọc theo độ dài trước
                cand = str1[:L]
                if cand * (n1 // L) == str1 and cand * (n2 // L) == str2:
                    return cand
        return ""

assert SolutionBrute().gcdOfStrings("ABABAB", "ABAB") == "AB"
assert SolutionBrute().gcdOfStrings("AAAAAB", "AAA") == ""
print("SolutionBrute PASS")

SolutionBrute PASS


## 8. Vì sao cách 2 đúng?

**(⇒)** Nếu có `x` chia hết cả hai: `str1 = x·p`, `str2 = x·q` ⇒ `str1 + str2 = x·(p+q) = str2 + str1`. Vậy khi hai cách ghép **khác nhau** thì chắc chắn không có ước chung → trả `""` đúng.

**(⇐)** Nếu `str1 + str2 == str2 + str1`: chạy Euclid trên chuỗi (cách 3) — điều kiện giao hoán đảm bảo chuỗi dài luôn bắt đầu bằng chuỗi ngắn, và đẳng thức giao hoán được **giữ nguyên** sau mỗi lần cắt (vì `(a - b) + b = b + (a - b)` khi `a = b + (a-b)` và `a + b = b + a`). Euclid dừng với một chuỗi `x` chia hết cả hai, có độ dài đúng `gcd(len1, len2)` (vì độ dài biến đổi y như Euclid trên số).

**Dài nhất:** mọi ước chung `y` có `len(y)` chia hết cả `len1`, `len2` ⇒ `len(y) <= gcd(len1, len2)`. Mà `x` dài đúng `gcd` ⇒ lớn nhất. Và ước chung có độ dài `g` thì phải là `str1[:g]`.

## 9. Độ phức tạp

Gọi `n1 = len(str1)`, `n2 = len(str2)`, `n = n1 + n2`.

| Cách | Time | Space phụ |
|---|---|---|
| 1. Thử mọi tiền tố | `O(min(n1,n2) · n)` | `O(n)` (chuỗi lặp tạm) |
| 1'. Lọc theo ước của độ dài | `O(d · n)`, `d` = số ước chung (nhỏ) | `O(n)` |
| 2. Giao hoán + GCD | `O(n)` (ghép + so sánh) + `O(log n)` cho gcd | `O(n)` |
| 3. Euclid trên chuỗi | worst `O(n²)` do cắt chuỗi tạo bản sao mới (ví dụ `"A"*1000` và `"A"`) | `O(n)` |

Cách 2 là tốt nhất: tuyến tính và cực ngắn.

## 10. Bộ test

Nhóm test:
1. Bốn ví dụ đề.
2. Hai chuỗi bằng nhau: đáp án là chính chuỗi.
3. Một chuỗi một ký tự.
4. GCD độ dài nhỏ hơn chuỗi ngắn: `"AAAAAA"`, `"AAAA"` → `"AA"`.
5. Độ dài nguyên tố cùng nhau nhưng cùng ký tự: `"AAA"`, `"AA"` → `"A"`.
6. Khối giống nhưng lệch đuôi: `"ABAB"`, `"ABA"` → `""`.
7. Đối chiếu ngẫu nhiên 4 cách.

In [17]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:24} PASS -> {actual!r}")

tests = [
    ("đề 1", "ABCABC", "ABC", "ABC"),
    ("đề 2", "ABABAB", "ABAB", "AB"),
    ("đề 3", "LEET", "CODE", ""),
    ("đề 4", "AAAAAB", "AAA", ""),
    ("bằng nhau", "XYZ", "XYZ", "XYZ"),
    ("1 ký tự", "A", "AAAA", "A"),
    ("gcd < chuỗi ngắn", "AAAAAA", "AAAA", "AA"),
    ("độ dài nguyên tố cùng nhau", "AAA", "AA", "A"),
    ("lệch đuôi", "ABAB", "ABA", ""),
    ("đảo thứ tự tham số", "ABC", "ABCABC", "ABC"),
]
for name, s1, s2, exp in tests:
    check(name, gcd_strings_concat(s1, s2), exp)

đề 1                     PASS -> 'ABC'
đề 2                     PASS -> 'AB'
đề 3                     PASS -> ''
đề 4                     PASS -> ''
bằng nhau                PASS -> 'XYZ'
1 ký tự                  PASS -> 'A'
gcd < chuỗi ngắn         PASS -> 'AA'
độ dài nguyên tố cùng nhau PASS -> 'A'
lệch đuôi                PASS -> ''
đảo thứ tự tham số       PASS -> 'ABC'


In [18]:
import random
random.seed(1071)
for _ in range(3000):
    if random.random() < 0.6:                     # thường sinh cặp CÓ ước chung
        base = "".join(random.choice("AB") for _ in range(random.randint(1, 3)))
        s1 = base * random.randint(1, 6)
        s2 = base * random.randint(1, 6)
    else:
        s1 = "".join(random.choice("AB") for _ in range(random.randint(1, 8)))
        s2 = "".join(random.choice("AB") for _ in range(random.randint(1, 8)))
    r = [gcd_strings_brute(s1, s2), gcd_strings_concat(s1, s2),
         gcd_strings_euclid(s1, s2), gcd_strings_euclid_iter(s1, s2)]
    assert len(set(r)) == 1, (s1, s2, r)
print("3000 test ngẫu nhiên: 4 cách khớp nhau")

3000 test ngẫu nhiên: 4 cách khớp nhau


## 11. Lỗi người mới hay gặp

### Lỗi 1: Trả về tiền tố `gcd` mà **quên** kiểm tra giao hoán

In [19]:
def wrong_no_check(str1, str2):
    return str1[:math.gcd(len(str1), len(str2))]

print("sai :", repr(wrong_no_check("LEET", "CODE")), "| đúng:", repr(gcd_strings_concat("LEET", "CODE")))

sai : 'LEET' | đúng: ''


### Lỗi 2: Lấy chuỗi ngắn hơn làm đáp án

Chỉ đúng khi chuỗi ngắn chia hết chuỗi dài (`"ABCABC"`, `"ABC"`). Với `"ABABAB"`, `"ABAB"` đáp án là `"AB"`, không phải `"ABAB"`.

### Lỗi 3: Chỉ kiểm tra "tiền tố chung", quên kiểm tra **lặp lại khít**

`"A"` là tiền tố chung của `"ABAB"` và `"ABA"`, độ dài 1 chia hết cả 4 và 3 — nhưng `"ABAB"` đâu phải `"A"` lặp lại. Tương tự `"AB"` với `"ABCD"`, `"ABXY"`.

In [20]:
def wrong_prefix_only(str1, str2):
    for L in range(min(len(str1), len(str2)), 0, -1):
        if str1[:L] == str2[:L] and len(str1) % L == 0 and len(str2) % L == 0:
            return str1[:L]            # thiếu kiểm tra lặp lại
    return ""

print("sai :", repr(wrong_prefix_only("ABAB", "ABA")), "| đúng:", repr(gcd_strings_concat("ABAB", "ABA")))
print("sai :", repr(wrong_prefix_only("ABCD", "ABXY")), "| đúng:", repr(gcd_strings_concat("ABCD", "ABXY")))

sai : 'A' | đúng: ''
sai : 'AB' | đúng: ''


### Lỗi 4: Thử tiền tố từ **ngắn** đến dài rồi trả ngay

Sẽ trả ước chung **nhỏ nhất** (`"A"` cho `"AAAAAA"`, `"AAAA"`) thay vì lớn nhất `"AA"`. Hoặc thử từ ngắn nhưng nhớ cập nhật đến hết.

### Lỗi 5: Đệ quy Euclid trên chuỗi dài → `RecursionError`

Đã thấy ở mục 6: dùng vòng lặp thay đệ quy khi số lần gọi có thể vượt ~1000.

## 12. Mẹo nhận dạng pattern

- "Chuỗi là lặp của một khối" → nghĩ tới **`s + t == t + s`**, hoặc mẹo `s in (s + s)[1:-1]` (bài 459).
- Hai đại lượng cùng phải "chia hết" theo một đơn vị → **GCD** của các độ dài/số.
- Euclid không chỉ dùng cho số: mọi cấu trúc có phép "trừ" hợp lý (chuỗi, đoạn thẳng, hình chữ nhật cắt hình vuông) đều áp dụng được.

| Bài liên quan | Ý tưởng |
|---|---|
| 459. Repeated Substring Pattern | `s` có là lặp của khối con? |
| 914. X of a Kind in a Deck of Cards | GCD của các tần suất |
| 1979. Find GCD of Array | `gcd(min, max)` |
| 365. Water and Jug Problem | GCD (định lý Bézout) |

## 13. Biến thể

### Biến thể A — Chuỗi có là lặp của một khối ngắn hơn không? (LeetCode 459)

Mẹo: `s` là lặp của khối ngắn hơn ⇔ `s` xuất hiện trong `(s + s)` sau khi bỏ ký tự đầu và cuối.

In [21]:
def repeated_substring_pattern(s: str) -> bool:
    return s in (s + s)[1:-1]

for s in ["abab", "aba", "abcabcabc", "a"]:
    print(repr(s), repeated_substring_pattern(s))

'abab' True
'aba' False
'abcabcabc' True
'a' False


### Biến thể B — Khối cơ sở nhỏ nhất (primitive root) của một chuỗi

Khối ngắn nhất `t` mà `s = t·k`. Dùng ước của `len(s)` từ nhỏ đến lớn.

In [22]:
def primitive_root(s: str) -> str:
    n = len(s)
    for L in range(1, n + 1):
        if n % L == 0 and s[:L] * (n // L) == s:
            return s[:L]
    return s

print(primitive_root("ABABAB"), primitive_root("AAAA"), primitive_root("ABC"))

AB A ABC


## 14. Bài tập tự luyện (tăng dần)

**Bài 1 (tính tay):** `gcd(84, 36)` theo Euclid; rồi đáp án của `str1 = "XY"*42`, `str2 = "XY"*18` dài bao nhiêu ký tự?

**Bài 2 (dự đoán output):** `"AB" + "ABAB" == "ABAB" + "AB"`? Còn `"AB" + "ABA" == "ABA" + "AB"`?

**Bài 3 (điền chỗ trống):** `ex_gcd_strings(str1, str2)` theo cách 2.

**Bài 4 (viết lại):** `ex_gcd_of_list(strs)` — ước chung lớn nhất của **cả một list** chuỗi (gợi ý: gộp dần như `gcd(gcd(a, b), c)`).

**Bài 5 (mini challenge):** `ex_lcm_strings(s1, s2)` — chuỗi **ngắn nhất** chia hết bởi cả `s1` và `s2` ("bội chung nhỏ nhất"), hoặc `""` nếu không tồn tại. Gợi ý: `lcm(a, b) = a * b // gcd(a, b)`.

In [23]:
def ex_gcd_strings(str1: str, str2: str) -> str:
    # TODO 1: nếu str1 + str2 khác str2 + str1 -> ""
    # TODO 2: g = math.gcd(...)
    # TODO 3: trả tiền tố độ dài g
    pass

# assert ex_gcd_strings("ABABAB", "ABAB") == "AB"
# assert ex_gcd_strings("LEET", "CODE") == ""

In [24]:
def ex_gcd_of_list(strs: list[str]) -> str:
    # TODO: res = strs[0]; với mỗi s còn lại: res = gcd_strings(res, s); nếu res == "" thì dừng
    pass

# assert ex_gcd_of_list(["ABABAB", "ABAB", "AB"]) == "AB"
# assert ex_gcd_of_list(["AA", "AAA", "B"]) == ""

In [25]:
def ex_lcm_strings(s1: str, s2: str) -> str:
    # TODO: không có ước chung -> ""; ngược lại lặp khối g để đạt độ dài lcm
    pass

# assert ex_lcm_strings("ABAB", "ABABAB") == "ABABABABABAB"
# assert ex_lcm_strings("AB", "BA") == ""

## 15. Gợi ý 3 mức

**Bài 1:** (1) `gcd(a, b) = gcd(b, a % b)`. (2) `(84,36) → (36,12) → (12,0)`. (3) GCD = 12; độ dài chuỗi `84`, `36` → đáp án dài `12` ký tự (`"XY"*6`).

**Bài 2:** (1) Ghép thử. (2) `"ABABAB"` vs `"ABABAB"`; `"ABABA"` vs `"ABAAB"`. (3) `True`, `False`.

**Bài 3:** (1) Như mục 5. (2) `math.gcd(len(str1), len(str2))`. (3) `return str1[:g]`.

**Bài 4:** (1) GCD có tính kết hợp. (2) Dùng lại hàm bài 3 hoặc `gcd_strings_concat`. (3) Vòng `for s in strs[1:]`.

**Bài 5:** (1) Có ước chung `x` thì mọi thứ là lặp của `x`. (2) `L = n1 * n2 // gcd(n1, n2)`. (3) `return x * (L // len(x))`.

## 16. Đáp án tham khảo

Tự làm trước rồi mới xem. Bài 1: `gcd(84, 36) = 12`, đáp án dài 12. Bài 2: `True`, `False`.

In [26]:
def ans_gcd_strings(str1, str2):
    if str1 + str2 != str2 + str1:
        return ""
    return str1[:math.gcd(len(str1), len(str2))]

def ans_gcd_of_list(strs):
    res = strs[0]
    for s in strs[1:]:
        res = ans_gcd_strings(res, s)
        if res == "":
            break
    return res

def ans_lcm_strings(s1, s2):
    x = ans_gcd_strings(s1, s2)
    if not x:
        return ""
    L = len(s1) * len(s2) // math.gcd(len(s1), len(s2))
    return x * (L // len(x))

assert math.gcd(84, 36) == 12 and len(ans_gcd_strings("XY" * 42, "XY" * 18)) == 12
assert ("AB" + "ABAB" == "ABAB" + "AB") is True and ("AB" + "ABA" == "ABA" + "AB") is False
assert ans_gcd_strings("ABABAB", "ABAB") == "AB" and ans_gcd_strings("LEET", "CODE") == ""
assert ans_gcd_of_list(["ABABAB", "ABAB", "AB"]) == "AB" and ans_gcd_of_list(["AA", "AAA", "B"]) == ""
assert ans_lcm_strings("ABAB", "ABABAB") == "ABABABABABAB" and ans_lcm_strings("AB", "BA") == ""
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


## 17. Template nhớ nhanh + checklist submit

```python
import math

class Solution:
    def gcdOfStrings(self, str1: str, str2: str) -> str:
        if str1 + str2 != str2 + str1:      # không cùng một khối lặp
            return ""
        return str1[:math.gcd(len(str1), len(str2))]
```

**Công thức nhớ:** *"Ghép hai chiều bằng nhau ⇒ cùng khối; độ dài khối lớn nhất = gcd độ dài."*

**Checklist:**
- [ ] Có kiểm tra `str1 + str2 == str2 + str1` trước khi cắt tiền tố.
- [ ] Dùng `gcd` của **độ dài**, không phải `min` độ dài.
- [ ] Trả `""` (chuỗi rỗng), không phải `None`.
- [ ] Test `"LEET","CODE"` → `""`, `"AAAAAA","AAAA"` → `"AA"`, `"ABAB","ABA"` → `""`.
- [ ] Time `O(n1 + n2)`, Space `O(n1 + n2)`.